# 04 — Sémantique : ontologie, RDF et SPARQL

Nos tables disent ce que contiennent les colonnes, pas ce que signifient les lignes. L'ontologie écrit ce sens :
un département est situé dans une région, une mesure de jour de froid est une mesure journalière, etc.

1. décrire le domaine avec une **ontologie OWL** (classes, relations, attributs) ;
2. y charger les régions, départements et résultats gold sous forme de **triplets RDF** ;
3. les interroger avec **SPARQL** (3 requêtes utiles au persona) ;
4. relier nos régions à **Wikidata** ;
5. visualiser le **graphe de connaissances**.

In [1]:
%pip install -q rdflib pyvis

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD

DOSSIER = "/home/jovyan/work/ontologie"
EN = Namespace("http://example.org/enedis#")       # l'espace de noms de notre projet
WD = Namespace("http://www.wikidata.org/entity/")  # Wikidata

g = Graph()
g.bind("en", EN); g.bind("owl", OWL); g.bind("wd", WD)

## 1. L'ontologie : le vocabulaire du domaine

On déclare les **classes** (les types de choses), les **relations** entre elles et les **attributs** (les valeurs).
Les sous-classes (⊂) permettent au graphe de faire des déductions : une mesure de jour de froid EST une mesure journalière.

In [3]:
def classe(nom, libelle, parent=None):
    g.add((EN[nom], RDF.type, OWL.Class))
    g.add((EN[nom], RDFS.label, Literal(libelle, lang="fr")))
    if parent:
        g.add((EN[nom], RDFS.subClassOf, EN[parent]))

def relation(nom, domaine, cible, libelle):
    g.add((EN[nom], RDF.type, OWL.ObjectProperty))
    g.add((EN[nom], RDFS.domain, EN[domaine]))
    g.add((EN[nom], RDFS.range, EN[cible]))
    g.add((EN[nom], RDFS.label, Literal(libelle, lang="fr")))

def attribut(nom, domaine, type_xsd, libelle):
    g.add((EN[nom], RDF.type, OWL.DatatypeProperty))
    g.add((EN[nom], RDFS.domain, EN[domaine]))
    g.add((EN[nom], RDFS.range, type_xsd))
    g.add((EN[nom], RDFS.label, Literal(libelle, lang="fr")))

# Classes (et sous-classes)
classe("Region", "région")
classe("Departement", "département")
classe("ProfilClient", "famille de clients")
classe("ProfilResidentiel", "clients résidentiels (RES)", parent="ProfilClient")
classe("ProfilProfessionnel", "clients professionnels (PRO)", parent="ProfilClient")
classe("ProfilEntreprise", "petites entreprises (ENT)", parent="ProfilClient")
classe("Mesure", "mesure de consommation")
classe("MesureJournaliere", "consommation d'une région sur un jour", parent="Mesure")
classe("MesureJourDeFroid", "consommation d'un des 10 jours les plus froids", parent="MesureJournaliere")
classe("MesurePointe", "puissance moyenne à une heure donnée", parent="Mesure")

# Relations entre objets
relation("situeDans", "Departement", "Region", "est situé dans")
relation("concerne", "Mesure", "Region", "concerne la région")
relation("pourProfil", "MesurePointe", "ProfilClient", "pour la famille de clients")

# Attributs (valeurs)
attribut("codeRegion", "Region", XSD.string, "code INSEE de la région")
attribut("nom", "Region", XSD.string, "nom")
attribut("codeInsee", "Departement", XSD.string, "code INSEE du département")
attribut("gradientMWParDegre", "Region", XSD.decimal, "thermosensibilité (MW par °C)")
attribut("gradientWParClient", "Region", XSD.decimal, "thermosensibilité par client (W par °C)")
attribut("rangReactivite", "Region", XSD.integer, "rang de réactivité au froid")
attribut("date", "MesureJournaliere", XSD.date, "date")
attribut("temperatureMoyenne", "MesureJournaliere", XSD.decimal, "température moyenne (°C)")
attribut("puissanceMoyenneMW", "Mesure", XSD.decimal, "puissance moyenne (MW)")
attribut("pointeMW", "MesureJournaliere", XSD.decimal, "pointe de la journée (MW)")
attribut("mois", "MesurePointe", XSD.integer, "mois (1 à 12)")
attribut("heure", "MesurePointe", XSD.integer, "heure (0 à 23)")

print(len(g), "triplets dans l'ontologie")

86 triplets dans l'ontologie


In [4]:
print(g.serialize(format="turtle")[:2500])

@prefix en: <http://example.org/enedis#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

en:MesureJourDeFroid a owl:Class ;
    rdfs:label "consommation d'un des 10 jours les plus froids"@fr ;
    rdfs:subClassOf en:MesureJournaliere .

en:ProfilEntreprise a owl:Class ;
    rdfs:label "petites entreprises (ENT)"@fr ;
    rdfs:subClassOf en:ProfilClient .

en:ProfilProfessionnel a owl:Class ;
    rdfs:label "clients professionnels (PRO)"@fr ;
    rdfs:subClassOf en:ProfilClient .

en:ProfilResidentiel a owl:Class ;
    rdfs:label "clients résidentiels (RES)"@fr ;
    rdfs:subClassOf en:ProfilClient .

en:codeInsee a owl:DatatypeProperty ;
    rdfs:label "code INSEE du département"@fr ;
    rdfs:domain en:Departement ;
    rdfs:range xsd:string .

en:codeRegion a owl:DatatypeProperty ;
    rdfs:label "code INSEE de la région"@fr ;
    rdfs:domain en:Region ;
    rdfs:range xsd:st

## 2. Les données : régions, départements, familles de clients

Les régions et leur thermosensibilité viennent de la table gold `gold_thermosensibilite_region` (PostgreSQL),
les départements du référentiel `ref/departements_regions.csv`.

In [7]:
import sys; sys.path.append("/home/jovyan/work")
from commun.donnees import lire

thermo = lire("gold_thermosensibilite_region")
ref = pd.read_csv("/home/jovyan/work/ref/departements_regions.csv", dtype=str, sep=";")
ref = ref[ref["code_region"].isin(thermo["code_region"].astype(str))]   # sans la Corse

print(len(thermo), "régions ;", len(ref), "départements")

12 régions ; 94 départements


In [8]:
def uri_region(code):
    return EN[f"region_{code}"]

for _, r in thermo.iterrows():
    u = uri_region(r.code_region)
    g.add((u, RDF.type, EN.Region))
    g.add((u, RDFS.label, Literal(r.nom_region, lang="fr")))
    g.add((u, EN.nom, Literal(r.nom_region)))
    g.add((u, EN.codeRegion, Literal(str(r.code_region))))
    g.add((u, EN.gradientMWParDegre, Literal(round(float(r.gradient_mw_par_degre), 1), datatype=XSD.decimal)))
    g.add((u, EN.gradientWParClient, Literal(round(float(r.gradient_w_par_client), 1), datatype=XSD.decimal)))
    g.add((u, EN.rangReactivite, Literal(int(r.rang_reactivite), datatype=XSD.integer)))

for _, d in ref.iterrows():
    u = EN[f"dep_{d.code_departement}"]
    g.add((u, RDF.type, EN.Departement))
    g.add((u, EN.codeInsee, Literal(d.code_departement)))
    g.add((u, EN.situeDans, uri_region(d.code_region)))

familles = {"RES": ("ProfilResidentiel", "résidentiels"),
            "PRO": ("ProfilProfessionnel", "professionnels"),
            "ENT": ("ProfilEntreprise", "petites entreprises")}
for code, (classe_famille, libelle) in familles.items():
    g.add((EN[f"famille_{code}"], RDF.type, EN[classe_famille]))
    g.add((EN[f"famille_{code}"], RDFS.label, Literal(libelle, lang="fr")))

print(len(g), "triplets au total")

458 triplets au total


## 3. Les mesures : résultats du gold

On ajoute les 4 380 mesures journalières (`gold_conso_jour_region`). Les 120 jours les plus froids (`gold_jours_froids`)
reçoivent le type plus précis `MesureJourDeFroid`. On ajoute aussi la puissance à 19 h (l'heure de pointe) par famille
de clients et par mois (`gold_pointe_horaire`).
Le graphe ne contient que des résultats agrégés : les 17 millions de lignes restent dans le data lake.

In [9]:
jours   = lire("gold_conso_jour_region")
froids  = lire("gold_jours_froids")
pointe19 = lire("SELECT * FROM gold_pointe_horaire WHERE heure = 19")

jours["d"]  = pd.to_datetime(jours["date"]).dt.strftime("%Y-%m-%d")
froids["d"] = pd.to_datetime(froids["date"]).dt.strftime("%Y-%m-%d")
cles_froides = set(zip(froids["code_region"].astype(str), froids["d"]))

for _, j in jours.iterrows():
    froid = (str(j.code_region), j.d) in cles_froides
    u = EN[f"jour_{j.code_region}_{j.d}"]
    g.add((u, RDF.type, EN.MesureJourDeFroid if froid else EN.MesureJournaliere))
    g.add((u, EN.concerne, uri_region(j.code_region)))
    g.add((u, EN.date, Literal(j.d, datatype=XSD.date)))
    g.add((u, EN.temperatureMoyenne, Literal(round(float(j.tmoy_c), 2), datatype=XSD.decimal)))
    g.add((u, EN.puissanceMoyenneMW, Literal(round(float(j.puissance_moy_mw), 1), datatype=XSD.decimal)))
    g.add((u, EN.pointeMW, Literal(round(float(j.pointe_mw), 1), datatype=XSD.decimal)))

for _, p in pointe19.iterrows():
    u = EN[f"pointe_{p.code_region}_{p.famille_profil}_mois{int(p.mois)}_19h"]
    g.add((u, RDF.type, EN.MesurePointe))
    g.add((u, EN.concerne, uri_region(p.code_region)))
    g.add((u, EN.pourProfil, EN[f"famille_{p.famille_profil}"]))
    g.add((u, EN.mois, Literal(int(p.mois), datatype=XSD.integer)))
    g.add((u, EN.heure, Literal(19, datatype=XSD.integer)))
    g.add((u, EN.puissanceMoyenneMW, Literal(round(float(p.puissance_moy_mw), 1), datatype=XSD.decimal)))

print(len(jours), "jours ;", len(froids), "jours froids ;", len(pointe19), "mesures de pointe à 19 h")
print(len(g), "triplets au total")

4380 jours ; 120 jours froids ; 432 mesures de pointe à 19 h
29330 triplets au total
